In [3]:
!pip install -U  trl datasets accelerate bitsandbytes sentencepiece jsonschema peft

In [ ]:
import csv
import json
from pathlib import Path

# === Input paths ===
CSV_PATH = Path("top_1000_task_messages.csv")
LABELS_PATH = Path("jira_tasks_first_1000_final.json")  # list of JSON dicts
OUTPUT_PATH = Path("train.jsonl")

# If your labels use a different field for matching, change this:
LABEL_MATCH_KEY = "issue_creation_date"   # e.g., "timestamp" if that's what your labels use

# === Step 1. Load labels and index by timestamp-like key ===
with LABELS_PATH.open("r", encoding="utf-8") as f:
    labels = json.load(f)

label_lookup = {}
for item in labels:
    key = item.get(LABEL_MATCH_KEY)
    if key is not None:
        label_lookup[str(key)] = item  # stringify to match CSV string timestamps

# === Step 2. Read Slack CSV and pair with labels ===
jsonl_records = []
with CSV_PATH.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        ts = str(row.get("timestamp", "")).strip()
        slack_text = (row.get("cleaned_text") or "").strip()
        if not ts or not slack_text:
            continue

        label = label_lookup.get(ts)
        if not label:
            continue  # skip if no matching label

        record = {
            "input": {
                "timestamp": ts,
                "text": slack_text
            },
            "output": label  # keep as JSON object (not a pretty-printed string)
        }
        jsonl_records.append(record)

# === Step 3. Write JSONL ===
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for rec in jsonl_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f" Saved {len(jsonl_records)} examples to {OUTPUT_PATH}")


✅ Saved 999 examples to train.jsonl


In [5]:
%pip install protobuf

Note: you may need to restart the kernel to use updated packages.


In [6]:
# train_qlora.py
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig, TrainingArguments)
from trl import SFTTrainer
from peft import LoraConfig
import json

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"   # swap for Llama-3 or Qwen2
DATA_FILES = {"train":"train.jsonl", "validation":"val.jsonl"}

# 4-bit loading (QLoRA)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype="bfloat16")

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tok.padding_side = "right"
tok.pad_token = tok.eos_token  # causal LM needs a pad

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map="auto"
)

# LoRA config
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
)




/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading checkpoint shards: 100%|██████████| 3/3 [00:12<00:00,  4.02s/it]


In [7]:
!pip install scikit-learn

In [8]:
from sklearn.model_selection import train_test_split
import json

# Load the generated records
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    jsonl_records = [json.loads(line) for line in f]

# Split the data into training and validation sets
train_records, val_records = train_test_split(jsonl_records, test_size=0.2, random_state=42) # 80% train, 20% validation

# Define validation output path
VAL_OUTPUT_PATH = "val.jsonl"

# Write the validation data to val.jsonl
# After train_test_split(...)
with open("train.jsonl", "w", encoding="utf-8") as f:
    for rec in train_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

with open("val.jsonl", "w", encoding="utf-8") as f:
    for rec in val_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


In [9]:
%pip install peft

Note: you may need to restart the kernel to use updated packages.


In [13]:
from datasets import load_dataset
import json

# Load dataset
ds = load_dataset("json", data_files=DATA_FILES)

# Build text from chat template: messages + gold output (as assistant)
def map_to_text(example):
    # Input is a dict: {"timestamp": "...", "text": "..."}
    inp = example["input"]
    if isinstance(inp, dict):
        ts = str(inp.get("timestamp", "")).strip()
        msg = str(inp.get("text", "")).strip()
        # Create a single user string including timestamp + text
        user_content = f"[{ts}] {msg}" if ts else msg
    else:
        # Back-compat if some rows are still strings
        user_content = str(inp)

    # Output may be a dict — stringify deterministically
    out = example["output"]
    if not isinstance(out, str):
        # compact but stable; chat templates usually don't need pretty printing
        out = json.dumps(out, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

    convo = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": out},
    ]
    text = tok.apply_chat_template(
        convo, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

# Map to a single 'text' column
ds = ds.map(
    map_to_text,
    remove_columns=ds["train"].column_names,  # drop original columns
    desc="Formatting with chat template",
)

# Training args (use 'evaluation_strategy', not 'eval_strategy')
args = TrainingArguments(
    output_dir="./slack2jira-qlora",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=3,
    logging_steps=20,
    eval_strategy="steps",   # <-- fix name
    eval_steps=200,
    save_steps=200,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    peft_config=lora,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    # dataset_text_field="text",     # <-- required since you prebuilt 'text'
    # max_seq_length=2048,
    # packing=False,
    args=args,
)

trainer.train()
trainer.save_model("./slack2jira-qlora")
tok.save_pretrained("./slack2jira-qlora")


Truncating eval dataset: 100%|██████████| 200/200 [00:00<00:00, 101226.11 examples/s]


Step,Training Loss,Validation Loss


('./slack2jira-qlora/tokenizer_config.json',
 './slack2jira-qlora/special_tokens_map.json',
 './slack2jira-qlora/chat_template.jinja',
 './slack2jira-qlora/tokenizer.model',
 './slack2jira-qlora/added_tokens.json',
 './slack2jira-qlora/tokenizer.json')

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch, re, json, datetime as dt

BASE = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER = "./slack2jira-qlora"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype="bfloat16")
tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER)

SYSTEM = "Extract a Jira ticket JSON from a Slack message. If no clear action, return {\"no_action\": true}. Output JSON only."
def predict(slack_text: str):
    SYSTEM = "Extract a Jira ticket JSON from a Slack message. If no clear action, return a JSON object with a field no_action set to true. Output JSON only."
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": slack_text}
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)

    # Generate
    out = model.generate(**ids, max_new_tokens=256, do_sample=False)

    # ✅ Decode ONLY the generated continuation (not the prompt)
    prompt_len = ids["input_ids"].shape[1]
    gen_tokens = out[0][prompt_len:]
    text = tok.decode(gen_tokens, skip_special_tokens=True)

    # ✅ Non-greedy JSON; take the LAST match
    m_all = list(re.finditer(r"\{.*?\}", text, flags=re.S))
    if not m_all:
        return {"no_action": True}
    try:
        obj = json.loads(m_all[-1].group(0))
    except Exception:
        return {"no_action": True}

    # Keep your post-processing
    if obj.get("no_action") is True:
        return {"no_action": True}

    obj.setdefault("task_summary", "")
    obj["task_summary"] = obj["task_summary"][:120]
    obj.setdefault("assignee", "")
    obj.setdefault("issue_creation_date", dt.datetime.utcnow().isoformat() + "Z")
    return obj


print(predict("<@Ketki> Can you make the discussed changes in the final project slides?"))


Loading checkpoint shards: 100%|██████████| 3/3 [00:26<00:00,  8.79s/it]
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{'assignee': 'Ketki', 'issue_creation_date': '2018-09-10T00:00:00.000100', 'task_summary': 'Make changes in final project slides'}


In [17]:
import re, json, datetime as dt
from typing import Optional, Dict, Any

NULL_OUT = {"task_summary": None, "assignee": None, "issue_creation_date": None}

def _build_user_string(slack_input) -> str:
    # Accept {"timestamp","text"} or a plain string
    if isinstance(slack_input, dict):
        ts = str(slack_input.get("timestamp", "")).strip()
        msg = str(slack_input.get("text", "")).strip()
        return f"[{ts}] {msg}" if ts else msg
    return str(slack_input)

def _last_balanced_json(text: str) -> Optional[str]:
    """Return the last top-level JSON object substring via brace balancing, or None."""
    start = None
    depth = 0
    spans = []
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    spans.append((start, i + 1))
                    start = None
    if not spans:
        return None
    s, e = spans[-1]
    return text[s:e]

def _parse_json_strict(gen_text: str) -> Optional[Dict[str, Any]]:
    """Single-attempt parse: extract last balanced object and json.loads exactly once."""
    if not gen_text:
        return None
    cand = _last_balanced_json(gen_text)
    if cand is None:
        return None
    try:
        return json.loads(cand)
    except Exception:
        return None

def _project_to_schema(obj: Dict[str, Any]) -> Dict[str, Any]:
    """Return only the three fields, or nulls if invalid or no_action."""
    if not isinstance(obj, dict):
        return NULL_OUT
    if obj.get("no_action") is True:
        return NULL_OUT

    ts = obj.get("task_summary", None)
    asg = obj.get("assignee", None)
    dt_str = obj.get("issue_creation_date", None)

    # Strict: only accept strings or nulls; anything else → null
    ts = ts if isinstance(ts, str) or ts is None else None
    asg = asg if isinstance(asg, str) or asg is None else None
    dt_str = dt_str if isinstance(dt_str, str) or dt_str is None else None

    # If all three end up null, return all nulls
    if ts is None and asg is None and dt_str is None:
        return NULL_OUT

    return {"task_summary": ts, "assignee": asg, "issue_creation_date": dt_str}

def predict(slack_input) -> Dict[str, Any]:
    user_str = _build_user_string(slack_input)

    messages = [
        {"role": "system", "content":
         'Output ONLY a single JSON object with keys: '
         '"task_summary", "assignee", "issue_creation_date"; '
         'or {"no_action": true} if no clear action. No extra text.'},
        {"role": "user", "content": user_str}
    ]

    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **ids,
        max_new_tokens=256,
        do_sample=False,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
    )

    # Decode ONLY continuation
    prompt_len = ids["input_ids"].shape[1]
    gen_tokens = out[0][prompt_len:]
    text = tok.decode(gen_tokens, skip_special_tokens=True)

    # Strict, single-shot parse
    obj = _parse_json_strict(text)
    if obj is None:
        return NULL_OUT

    return _project_to_schema(obj)

# Example:
print(predict({"timestamp":"2019-01-30T19:57:20.606600","text":"<@Ketki> Can you make the discussed changes in the final project slides?"}))


{'task_summary': 'Make changes in final project slides', 'assignee': 'Ketki', 'issue_creation_date': '2019-01-30T19:57:20.606600'}


In [48]:
import re, json
from typing import Optional, Dict, Any

# === Desired "failure" shape helper ===
def _null_with_ts(ts_str: Optional[str]) -> Dict[str, Any]:
    return {"task_summary": "null", "assignee": None, "issue_creation_date": ts_str}

def _build_user_string(slack_input) -> str:
    if isinstance(slack_input, dict):
        ts = str(slack_input.get("timestamp", "")).strip()
        msg = str(slack_input.get("text", "")).strip()
        return f"[{ts}] {msg}" if ts else msg
    return str(slack_input)

def _last_balanced_json(text: str) -> Optional[str]:
    start = None
    depth = 0
    spans = []
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    spans.append((start, i + 1))
                    start = None
    if not spans:
        return None
    s, e = spans[-1]
    return text[s:e]

def _parse_json_strict(gen_text: str) -> Optional[Dict[str, Any]]:
    if not gen_text:
        return None
    cand = _last_balanced_json(gen_text)
    if cand is None:
        return None
    try:
        return json.loads(cand)
    except Exception:
        return None

def _project_to_schema(obj: Dict[str, Any], ts_fallback: Optional[str]) -> Dict[str, Any]:
    # If it's not a dict or model says no_action, force the "null" summary shape with input ts
    if not isinstance(obj, dict) or obj.get("no_action") is True:
        return _null_with_ts(ts_fallback)

    # Pull fields
    task_summary = obj.get("task_summary", None)
    assignee = obj.get("assignee", None)
    issue_date = obj.get("issue_creation_date", None)

    # Coerce/validate types
    task_summary = task_summary if isinstance(task_summary, str) else None
    assignee = assignee if isinstance(assignee, str) and assignee.strip() else None
    issue_date = issue_date if isinstance(issue_date, str) and issue_date.strip() else ts_fallback

    # If we don't have a usable summary, set it to the *string* "null"
    if not task_summary or not task_summary.strip():
        task_summary = "null"

    return {
        "task_summary": task_summary,
        "assignee": assignee,
        "issue_creation_date": issue_date,
    }

def predict(slack_input) -> Dict[str, Any]:
    # Keep the input timestamp handy for fallback
    ts_fallback = None
    if isinstance(slack_input, dict):
        ts_fallback = str(slack_input.get("timestamp") or "").strip() or None

    user_str = _build_user_string(slack_input)

    messages = [
        {"role": "system", "content":
         'Output ONLY one JSON object with keys: "task_summary", "assignee", "issue_creation_date"; '
         'or {"no_action": true} if no clear action. No extra text.'},
        {"role": "user", "content": user_str}
    ]

    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **ids,
        max_new_tokens=256,
        do_sample=False,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
    )

    # Decode only continuation
    prompt_len = ids["input_ids"].shape[1]
    gen_tokens = out[0][prompt_len:]
    text = tok.decode(gen_tokens, skip_special_tokens=True)

    # Single strict parse; then projection with fallback timestamp
    obj = _parse_json_strict(text)
    if obj is None:
        return _null_with_ts(ts_fallback)

    return _project_to_schema(obj, ts_fallback)


In [20]:
import json
import random
from pathlib import Path
from textwrap import indent

# === CONFIG ===
TRAIN_PATH = Path("train.jsonl")
VAL_PATH = Path("val.jsonl")
NUM_SAMPLES = 5  # how many examples to show from each split

def load_jsonl(path, limit=None):
    """Load JSONL file and optionally truncate."""
    data = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            data.append(json.loads(line))
            if limit and len(data) >= limit:
                break
    return data

def canonical_json(obj):
    """Nicely formatted JSON string for readability."""
    return json.dumps(obj, ensure_ascii=False, indent=2, sort_keys=True)

def coerce_output_obj(out):
    """Return output as a Python dict regardless of whether it's already a dict or a JSON string."""
    if isinstance(out, dict):
        return out
    if isinstance(out, str):
        return json.loads(out)
    raise TypeError(f"Unsupported output type: {type(out)}")

def build_user_message(inp):
    """Return a single string for the user message, handling old and new schemas."""
    if isinstance(inp, dict):
        ts = str(inp.get("timestamp", "")).strip()
        msg = str(inp.get("text", "")).strip()
        if ts and msg:
            return f"[{ts}] {msg}"
        return msg or ts
    # fallback to old format: extract the part after the blank line if present
    s = str(inp)
    return s.split("\n\n", 1)[-1] if "\n\n" in s else s

def show_samples(data, name="dataset"):
    print(f"\n===== {name.upper()} SAMPLES (Actual vs Predicted) =====")
    if not data:
        print("  (no examples)")
        return
    examples = random.sample(data, min(NUM_SAMPLES, len(data)))
    for i, ex in enumerate(examples, 1):
        # Parse input and output robustly
        slack_msg = build_user_message(ex.get("input"))
        ref_obj = coerce_output_obj(ex.get("output"))

        # Run your model
        pred_obj = predict(slack_msg)

        print(f"\n--- Sample #{i} ---")
        print(f"🗨️ Slack message:\n{indent((slack_msg or '').strip(), '  ')}")
        print(f"\n✅ Reference Jira JSON:")
        print(indent(canonical_json(ref_obj), "  "))
        print(f"\n🤖 Predicted Jira JSON:")
        print(indent(canonical_json(pred_obj), "  "))

        # Optional: compare task_summary field if available
        ref_summary = (ref_obj or {}).get("task_summary", "")
        pred_summary = (pred_obj or {}).get("task_summary", "")
        if ref_summary or pred_summary:
            print("\n📝 task_summary comparison:")
            print(f"  REF: {ref_summary}")
            print(f"  PRED: {pred_summary}")

# === Run the comparison ===
train_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)

show_samples(train_data, "train")
show_samples(val_data, "val")



===== TRAIN SAMPLES (Actual vs Predicted) =====

--- Sample #1 ---
🗨️ Slack message:
  [2017-07-23T08:53:33.191298] <@Shara> Yeah that’s ok, its what I do at the moment. The only nuisance with this is that the main update function will have a lot of “cases”

✅ Reference Jira JSON:
  {
    "assignee": "Shara",
    "issue_creation_date": "2017-07-23T08:53:33.191298",
    "task_summary": "Refactor main update function to reduce number of cases."
  }

🤖 Predicted Jira JSON:
  {
    "assignee": "Shara",
    "issue_creation_date": "2017-07-23T08:53:33.191298",
    "task_summary": "Refactor main update function to reduce number of cases."
  }

📝 task_summary comparison:
  REF: Refactor main update function to reduce number of cases.
  PRED: Refactor main update function to reduce number of cases.

--- Sample #2 ---
🗨️ Slack message:
  [2017-12-22T18:26:33.000144] <@Porsha>, do you happen to know where the code that has this bug is? Maybe I'll work on fix it.

✅ Reference Jira JSON:
  {
    "

In [22]:
# === ROUGE evaluation (fixed for dict outputs and new input schema) ===
# pip install -U evaluate rouge-score tqdm

import json
from pathlib import Path
from typing import Dict, Any, List, Tuple
from tqdm import tqdm
import evaluate

VAL_PATH = Path("val.jsonl")  # adjust if needed

def canonical_json_text(obj: Dict[str, Any]) -> str:
    """Deterministically stringify JSON for fair text comparison."""
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def coerce_output_obj(out) -> Dict[str, Any]:
    """Ensure we always have a dict for the reference."""
    if isinstance(out, dict):
        return out
    if isinstance(out, str):
        return json.loads(out)
    raise TypeError(f"Unsupported output type for reference: {type(out)}")

def get_task_summary_safe(obj: Dict[str, Any]) -> str:
    val = obj.get("task_summary")
    return val if isinstance(val, str) else ""

def run_inference_on_val(val_path: Path) -> Tuple[List[str], List[str], List[str], List[str]]:
    preds_full, refs_full = [], []
    preds_summary, refs_summary = [], []

    with val_path.open("r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Scoring ROUGE on val"):
            ex = json.loads(line)

            # Reference (already a dict in your JSONL format)
            ref_obj = coerce_output_obj(ex["output"])
            ref_full_text = canonical_json_text(ref_obj)

            # Prediction: pass the whole input dict to your predict() (it accepts dict or str)
            slack_input = ex["input"]   # {"timestamp": "...", "text": "..."}
            pred_obj = predict(slack_input)  # uses your strict parser + projection

            # Canonicalize predicted JSON
            pred_full_text = canonical_json_text(pred_obj)

            preds_full.append(pred_full_text)
            refs_full.append(ref_full_text)

            # Field-level (task_summary) ROUGE
            preds_summary.append(get_task_summary_safe(pred_obj))
            refs_summary.append(get_task_summary_safe(ref_obj))

    return preds_full, refs_full, preds_summary, refs_summary

def compute_rouge(preds: List[str], refs: List[str]) -> Dict[str, float]:
    rouge = evaluate.load("rouge")
    return rouge.compute(predictions=preds, references=refs)

if __name__ == "__main__":
    preds_full, refs_full, preds_sum, refs_sum = run_inference_on_val(VAL_PATH)

    print("\n=== ROUGE on full JSON ===")
    scores_full = compute_rouge(preds_full, refs_full)
    for k, v in scores_full.items():
        print(f"{k}: {v:.4f}")

    print("\n=== ROUGE on task_summary (field-level) ===")
    # Filter out pairs where both sides are empty
    filtered = [(p, r) for p, r in zip(preds_sum, refs_sum) if (p or r)]
    if filtered:
        p_sum, r_sum = zip(*filtered)
        scores_sum = compute_rouge(list(p_sum), list(r_sum))
        for k, v in scores_sum.items():
            print(f"{k}: {v:.4f}")
    else:
        print("No task_summary values found in predictions or references.")


Scoring ROUGE on val: 200it [12:20,  3.70s/it]



=== ROUGE on full JSON ===
rouge1: 0.8517
rouge2: 0.7737
rougeL: 0.8425
rougeLsum: 0.8421

=== ROUGE on task_summary (field-level) ===
rouge1: 0.5625
rouge2: 0.3549
rougeL: 0.5384
rougeLsum: 0.5375


In [23]:
import json
from pathlib import Path
from tqdm import tqdm
from typing import List, Dict, Any

VAL_PATH = Path("val.jsonl")

FIELDS = ["task_summary", "assignee", "issue_creation_date"]

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load JSONL file into a list of dicts."""
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def compute_prf1(tp, fp, fn):
    """Return precision, recall, f1."""
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1

def is_empty(val: Any) -> bool:
    """
    Treat None, 'null', '', 'None', or whitespace-only strings as empty.
    """
    if val is None:
        return True
    if isinstance(val, str):
        v = val.strip().lower()
        return v in ("", "null", "none")
    return False

def evaluate_fields(val_path: Path):
    data = load_jsonl(val_path)

    # Initialize counters
    results = {field: {"tp": 0, "fp": 0, "fn": 0} for field in FIELDS}

    for ex in tqdm(data, desc="Evaluating fields"):
        ref = ex["output"]
        pred = predict(ex["input"])  # uses your predict() function

        for field in FIELDS:
            ref_val = ref.get(field)
            pred_val = pred.get(field)

            ref_empty = is_empty(ref_val)
            pred_empty = is_empty(pred_val)

            # Case 1: both empty → correct but not counted as TP (no positive ground truth)
            if ref_empty and pred_empty:
                continue

            # Case 2: both non-empty
            if not ref_empty and not pred_empty:
                if str(ref_val).strip().lower() == str(pred_val).strip().lower():
                    results[field]["tp"] += 1
                else:
                    results[field]["fp"] += 1
                    results[field]["fn"] += 1

            # Case 3: reference empty but model predicted something → false positive
            elif ref_empty and not pred_empty:
                results[field]["fp"] += 1

            # Case 4: reference non-empty but model predicted empty → false negative
            elif not ref_empty and pred_empty:
                results[field]["fn"] += 1

    # Compute precision, recall, f1 per field
    summary = {}
    for field, counts in results.items():
        tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
        p, r, f1 = compute_prf1(tp, fp, fn)
        summary[field] = {"precision": p, "recall": r, "f1": f1}

    # Macro average (mean across fields)
    avg_p = sum(v["precision"] for v in summary.values()) / len(FIELDS)
    avg_r = sum(v["recall"] for v in summary.values()) / len(FIELDS)
    avg_f1 = sum(v["f1"] for v in summary.values()) / len(FIELDS)
    summary["macro_avg"] = {"precision": avg_p, "recall": avg_r, "f1": avg_f1}

    return summary


if __name__ == "__main__":
    scores = evaluate_fields(VAL_PATH)
    print("\n=== Precision / Recall / F1 per field ===")
    for field, vals in scores.items():
        print(f"{field:22s}  P={vals['precision']:.3f}  R={vals['recall']:.3f}  F1={vals['f1']:.3f}")


Evaluating fields: 100%|██████████| 200/200 [12:56<00:00,  3.88s/it]


=== Precision / Recall / F1 per field ===
task_summary            P=0.094  R=0.095  F1=0.094
assignee                P=0.938  R=0.945  F1=0.941
issue_creation_date     P=1.000  R=1.000  F1=1.000
macro_avg               P=0.677  R=0.680  F1=0.679


In [24]:
# Save the fine-tuned model and tokenizer
trainer.save_model("./slack2jira-qlora")
tok.save_pretrained("./slack2jira-qlora")

print("✅ Fine-tuned model and tokenizer saved.")

✅ Fine-tuned model and tokenizer saved.


###Update Jira Issue

In [34]:
!pip install python-dotenv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [35]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
"""
Strict Slack -> QLoRA -> Jira (create OR update), with no fallbacks.

Prereqs:
  pip install requests python-dotenv

Env vars:
  JIRA_BASE_URL=https://<site>.atlassian.net
  JIRA_EMAIL=you@example.com
  JIRA_API_TOKEN=...
  JIRA_PROJECT_KEY=KAN
  JIRA_ISSUE_TYPE=Task
  # Optional if you want to set assignee by accountId directly:
  JIRA_ASSIGNEE_ACCOUNT_ID=712020:7c4cf37a-d64a-4996-9058-554d93aaef95
"""

import os
import json
import requests
from requests.auth import HTTPBasicAuth
from typing import Dict, Any, Optional
from urllib.parse import urlparse


JIRA_BASE_URL     = os.getenv("JIRA_BASE_URL", "https://sjsuteam5.atlassian.net").strip()
JIRA_EMAIL        = (os.getenv("JIRA_EMAIL", "aishanee.sinha@sjsu.edu") or "").strip()
JIRA_API_TOKEN    = (os.getenv("JIRA_API_TOKEN") or "").strip()
JIRA_PROJECT_KEY  = os.getenv("JIRA_PROJECT_KEY", "KAN").strip()
JIRA_ISSUE_TYPE   = os.getenv("JIRA_ISSUE_TYPE", "Task").strip()  # Task, Story, Bug, etc.

def normalized_cloud_base(url: str) -> str:
    if not url:
        raise ValueError("JIRA_BASE_URL is empty.")
    if not url.startswith("http"):
        url = "https://" + url
    p = urlparse(url)
    if p.scheme != "https":
        raise ValueError("JIRA_BASE_URL must use https.")
    if not p.netloc.endswith(".atlassian.net"):
        raise ValueError("JIRA_BASE_URL must end with .atlassian.net")
    return f"{p.scheme}://{p.netloc}"

JIRA_BASE_URL = normalized_cloud_base(JIRA_BASE_URL)

def build_adf_paragraph(text: str) -> Dict[str, Any]:
    """Minimal Atlassian Document Format (ADF) doc."""
    return {
        "type": "doc",
        "version": 1,
        "content": [
            {"type": "paragraph", "content": [{"type": "text", "text": text}]}
        ],
    }

def require_env():
    if not (JIRA_EMAIL and JIRA_API_TOKEN):
        raise RuntimeError("JIRA_EMAIL or JIRA_API_TOKEN missing.")
    if not JIRA_PROJECT_KEY:
        raise RuntimeError("JIRA_PROJECT_KEY missing.")
    if not JIRA_ISSUE_TYPE:
        raise RuntimeError("JIRA_ISSUE_TYPE missing.")

def strict_from_model(slack_input: Dict[str, Any]) -> Dict[str, str]:
    """
    Call your QLoRA model's predict() exactly once and require fields.
    predict() must be imported/available from your fine-tune code.
    It must return: {"task_summary": str, "assignee": str|None, "issue_creation_date": str|None}
    No fallbacks here.
    """
    # predict() is the one you trained in your notebook / module (QLoRA).  # see fine-tune file
    out = predict(slack_input)
    print(out) # <-- provided by your QLoRA code
    # Validate strictly
    if not isinstance(out, dict):
        raise ValueError("Model output is not a JSON object.")
    if "task_summary" not in out:
        raise ValueError("Model output missing 'task_summary'.")
    task_summary = out.get("task_summary")
    assignee     = out.get("assignee")
    issue_date   = out.get("issue_creation_date")

    if not isinstance(task_summary, str) or not task_summary.strip():
        raise ValueError("Model 'task_summary' must be a non-empty string.")
    if assignee is not None and not isinstance(assignee, str):
        raise ValueError("Model 'assignee' must be a string or null.")
    if issue_date is not None and not isinstance(issue_date, str):
        raise ValueError("Model 'issue_creation_date' must be a string or null.")

    return {
        "task_summary": task_summary.strip(),
        "assignee": assignee.strip() if isinstance(assignee, str) else None,
        "issue_creation_date": issue_date.strip() if isinstance(issue_date, str) else None,
    }

def resolve_account_id(name_or_email: str) -> Optional[str]:
    """Optional: turn assignee name/email into accountId. Only used if model provides assignee."""
    try:
        u = f"{JIRA_BASE_URL}/rest/api/3/user/search"
        r = requests.get(u,
            params={"query": name_or_email},
            auth=HTTPBasicAuth(JIRA_EMAIL, JIRA_API_TOKEN),
            headers={"Accept": "application/json"},
            timeout=15,
        )
        r.raise_for_status()
        users = r.json()
        if isinstance(users, list) and users:
            return users[0].get("accountId")
    except Exception:
        pass
    return None

# ====== Jira API (Create / Update) ======
def create_issue(slack_input: Dict[str, Any]) -> str:
    """
    Create a Jira issue strictly from model output.
    Raises on any missing pieces. No fallbacks.
    """
    require_env()
    m = strict_from_model(slack_input)

    description_text = (
        "Source: Slack message\n"
        f"Timestamp: {str(slack_input.get('timestamp') or '')}\n\n"
        f"Original text:\n{str(slack_input.get('text') or '')}"
    )
    description_adf = build_adf_paragraph(description_text)

    fields: Dict[str, Any] = {
        "project": {"key": JIRA_PROJECT_KEY},
        "summary": m["task_summary"],
        "issuetype": {"name": JIRA_ISSUE_TYPE},
        "description": description_adf,
    }

    # Assignee only if model provided one (no defaulting)
    if m["assignee"]:
        acct = resolve_account_id(m["assignee"])
        if acct:
            fields["assignee"] = {"accountId": acct}

    url = f"{JIRA_BASE_URL}/rest/api/3/issue"
    resp = requests.post(
        url,
        json={"fields": fields},
        auth=HTTPBasicAuth(JIRA_EMAIL, JIRA_API_TOKEN),
        headers={"Accept": "application/json", "Content-Type": "application/json"},
        timeout=20,
    )
    try:
        resp.raise_for_status()
    except requests.HTTPError as e:
        raise RuntimeError(f"Create failed: {resp.status_code} {resp.text}") from e

    return resp.json().get("key")


# Your Slack input
slack_msg = {
    "timestamp": "2025-11-02T08:14:27.439600",
    "text": "<@Ketki> Can you make the discussed changes in the slack-jira agent code?"
}

#Create a new issue strictly from model output:
new_key = create_issue(slack_msg)
print("Created:", new_key)



{'task_summary': 'Make changes in slack-jira agent code', 'assignee': 'Ketki', 'issue_creation_date': '2025-11-02T08:14:27.439600'}
Created: KAN-26


In [ ]:

slack_msg = {
    "timestamp": "2025-11-02T08:14:27.439600",
    "text": "<@Debika> Can you fix the error in the intent-detection code?"
}

#Create a new issue strictly from model output:
new_key = create_issue(slack_msg)
print("Created:", new_key)


{'task_summary': 'Fix error in intent-detection code', 'assignee': 'Debika', 'issue_creation_date': '2025-11-02T08:14:27.439600'}
Created: KAN-27


In [ ]:

slack_msg = {
    "timestamp": "2025-11-02T08:14:27.439600",
    "text": "<@Soham> Can you update the slides for our mid-presentation?"
}

#Create a new issue strictly from model output:
new_key = create_issue(slack_msg)
print("Created:", new_key)


{'task_summary': 'Update slides for mid-presentation', 'assignee': 'Soham', 'issue_creation_date': '2025-11-02T08:14:27.439600'}
Created: KAN-28
